In [1]:
import os
import glob
import itertools
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from scipy.signal import chirp, find_peaks, peak_widths

functions

In [3]:
# 定义函数

def separate_session(df, threshold=3):
    df = pd.read_csv(df, index_col=0)
    df['index_col'] = pd.to_numeric(df.index, errors='coerce')
    check_points = [0]  # 初始化第一个分割点为0（DataFrame的起始位置）
    frame_list = df['index_col'].values.tolist()
    

    # 判断两行之间的时间差，大于threshold则视为新的session
    for i in range(len(frame_list) - 1):
        if (frame_list[i+1] - frame_list[i]) > threshold:
            check_points.append(frame_list[i+1])
    
    check_points.append(frame_list[-1] + 1)  # 添加最后一个分割点

    session_dfs = []
    for i in range(len(check_points) - 1):
        session_df = df[(df['index_col'] >= check_points[i]) & (df['index_col'] < check_points[i+1])]
        session_df.drop(columns='index_col', inplace=True)
        session_dfs.append(session_df)

    return session_dfs
   

# 处理分离后的session文件，使每个session的开始时间为0
def process_frame(df):
    # 获取DataFrame的index并转换为float类型的list
    index_list = [float(index) for index in df.index]

    # 判断第一个index的值是否为0，并进行相应处理
    if index_list[0] != 0:
        # 如果第一个index的值不为0，将所有index值减去第一个index值，以实现“相对于第一个index的增量”效果
        index_list = [index - index_list[0] for index in index_list]

    return index_list

  
def process_event_interval(df):
    # 从DataFrame中提取'From Second'和'To Second'列，并转换为二维列表
    event_list = df[['From Second', 'To Second']].values.tolist()

    # 返回处理后的新事件列表
    return event_list



# 获得事件标记
def get_event_label(event_df):
    
    '''
    Aim:
        for OFT
        translate 'event' in eventFile to integer label
        each row (corresponding to each time interval) of event is assigned a label
        called in function `add_event_label()`
    '''
    
    # get event status list
    Event_status = event_df['Event'].values.tolist()
    # compact event label list for three sessions
    event_label = []
    for i in Event_status:   
        if i.strip() == 'sniff':
            event_label.append(11)
        elif i.strip() == 'sniffed' :
            event_label.append(12)
        elif i.strip() == 'freezing':
            event_label.append(21)
        elif i.strip() == 'cs':
            event_label.append(22)
        elif i.strip() == 'In_all':
            event_label.append(23)
        else:
            event_label.append(24)

    return(event_label)


# 添加事件标签
def add_event_label(trace_df, event_path):

    '''
    Aim:
        add event_label to each frame
        one frame can have several label (since mouse can in a location and do something)
    
    Retrun:
        trace dataframe with one label column
    '''
    
    frame_list = process_frame(trace_df)
    #get event dataframe/eventIntervals/eventLabel
#     event_df = pd.read_csv(event_path)
    event_df = pd.read_excel(event_path)
    interval_list = process_event_interval(event_df)
    event_label = get_event_label(event_df)

    
    # add label to each frame
    frame_label_list = []
    for frame in frame_list:
        frame_label = str()
        
        for i in range(len(interval_list)): # one frame can be in several interval
            if min(interval_list[i])  <= frame <= max(interval_list[i]) :
                frame_label = frame_label+ str(event_label[i])

        # convert str to int
        if len(frame_label) == 0:
            frame_label = np.nan
        else:
            frame_label = int(frame_label)

        # append frame label to list
        frame_label_list.append(frame_label)
    
    
    # add label list to trace df 
    new_df = trace_df.copy()
    new_df['Frame_Label']  =  frame_label_list
    new_df = new_df.dropna()
    
    return(new_df)


In [9]:
# 检查特定状态下的帧间隔，并返回满足条件的数据子集
def check_frame_interval(df, which_status_to_check_frame):
    
    # set up which_status_to_check_frame
    
    if which_status_to_check_frame == 'sniff':
        which_status_to_check_frame = str(11)
    elif which_status_to_check_frame == 'sniffed':
        which_status_to_check_frame = str(12)
    elif which_status_to_check_frame == 'freezing':
        which_status_to_check_frame = str(21)
    elif which_status_to_check_frame == 'cs':
        which_status_to_check_frame = str(22)
    elif which_status_to_check_frame == 'In_all':
        which_status_to_check_frame = str(23)
    else:
        raise ValueError('Not approved status. choose from: Investigating_RIGHT_SNIFF, Investigating_LEFT_SNIFF, In_right, In_all, In_left')

    #df = add_event_label(df, event_1_path, event_2_path, event_3_path, mice_path)[which_session]
    frame_label_list = df['Frame_Label'].values.tolist()
    
    # to separate status label, eg:20214111
    n = 2
    
    check_frame_label_list = []
    for line in frame_label_list:
        line = str(line)
        line_list= [line[i:i+n] for i in range(0, len(line), n)]
        if which_status_to_check_frame in line_list:
            check_frame_label_list.append(1)
        else:
            check_frame_label_list.append(0)       
            
    
            
    df['status'] = check_frame_label_list
    df_keep = df[ (df['status'] == 1)]
    df_keep = df_keep.drop('status', axis = 1)
    df_keep = df_keep.drop('Frame_Label', axis = 1)
    
    #print('the shape of the dataframe in the status is:', df_keep.shape)
    return(df_keep)



def similarity(x, y):
    # 将输入转化为numpy数组
    x = np.array(x, dtype=float)
    y = np.array(y, dtype=float)

    # 计算相似度
    simi = 2 * np.dot(x, y) / (np.linalg.norm(x) ** 2 + np.linalg.norm(y) ** 2)
    return simi

# shuffle
#----------------------------------------------------------------------------------
def random_shuffle(X):
    new_X = []
    for i in range(X.shape[1]):
        x = X[:, i].copy()
        shuffle(x)
        new_X.append(x)
    return np.vstack(new_X).T


# collect samples
#----------------------------------------------------------------------------------
def collect_samples(arr, sample_size, n_samples):
    samples = np.zeros((n_samples, sample_size), np.int32)

    for sample_n in range(0, n_samples):
        sample = get_sample(arr,
                            n_iter=sample_n,
                            sample_size=sample_size)
        samples[sample_n] = sample

    return samples


# get samples
#----------------------------------------------------------------------------------

def get_sample(arr, n_iter=None, sample_size=10, fast=True):
    return np.random.choice(arr, sample_size, replace=False)




def clsfy(B, C):
    # 将输入转化为numpy数组
    B = np.array(B, dtype=float)
    C = np.array(C, dtype=float)

    b = B.copy()
    simi = similarity(B, C)
    x = np.linalg.norm(b) ** 2 + np.linalg.norm(C) ** 2
    bs = collect_samples(b, len(b), 5000)
    a = 2 * np.dot(C, bs.T) / x
    if simi >= np.percentile(a, 99.75):
        neuron_type = 'ON'
    elif simi <= np.percentile(a, 0.25):
        neuron_type = 'OFF'
    else:
        neuron_type = 'Other'
    return simi, a, neuron_type



# to check on/off neuron of certain status
### in this case, we only check social(11)/obj(12)
#----------------------------------------------------------------------------------
def status_ONOFF(df, which_status_to_check_ON_OFF, ON_OFF, fps):
    
    # set up which_status_to_check_ON_OFF
    
    if which_status_to_check_ON_OFF == 'sniff':
        which_status_to_check_ON_OFF = str(11)
    elif which_status_to_check_ON_OFF == 'sniffed':
        which_status_to_check_ON_OFF = str(12)
    elif which_status_to_check_ON_OFF == 'freezing':
        which_status_to_check_ON_OFF= str(21)
    elif which_status_to_check_ON_OFF == 'cs':
        which_status_to_check_ON_OFF = str(22)
    elif which_status_to_check_ON_OFF == 'In_all':
        which_status_to_check_ON_OFF = str(23)
    else:
        raise ValueError('Not approved status. choose from: free_allogroom, sniff, rear_selfgroom, all')

    frame_label_list = df['Frame_Label'].values.tolist()
    
    # to separate status label, eg:20214111
    n = 2
    
    label_list = []
    for line in frame_label_list:
        line = str(line)
        line_list= [line[i:i+n] for i in range(0, len(line), n)]
        if which_status_to_check_ON_OFF in line_list:
            label_list.append(1)
        else:
            label_list.append(0)
            
    label_list_array = np.asarray(label_list)
    
    # get dataframe array; drop last column: frame_label
    df_array = df.iloc[:, :-1].to_numpy()
    # get n_neuron
    T, n_neuron = df.iloc[:, :-1].shape
    
    # get on/off neuron type
    neuron_type_list = []
    if sum(label_list_array == 1) / fps >= 4:
        for i in range(n_neuron):
            simi, a, t = clsfy(label_list_array, df_array[:, i])
            neuron_type_list.append(t)
            
    # get on/off neurom dataframe
    df_columns  = df.iloc[:, :-1].columns
    new_df_column = []
    for i in range(len(neuron_type_list)):
        if neuron_type_list[i] == ON_OFF:
            neuron_keep = df_columns[i]
            new_df_column.append(neuron_keep)
            
    new_df = df[new_df_column]
    new_df['Frame_Label'] = frame_label_list
    
    # shape(frames, keep_neurons+frame_Label)
    return(new_df, new_df_column)

# 修改后的generate_ONOFF_Summary函数，返回df_ONOFF_summary而不是保存为csv文件
def generate_ONOFF_Summary(df, all_ONOFF_neuron_list, columns_name):
        
    all_neuron_list = df.columns.to_list()
    if 'Frame_Label' in all_neuron_list:
        all_neuron_list.remove('Frame_Label')
    
    label_list = []
    for i in all_neuron_list:
        label = []
        for a in range(len(all_ONOFF_neuron_list)):
            if i in all_ONOFF_neuron_list[a]:
                label.append(1)
            else:
                label.append(0)
        label_list.append(label)
        
    dic_ONOFF_summary = {}
    for i in range(len(all_neuron_list)):
        dic_ONOFF_summary[all_neuron_list[i]] = label_list[i]

    df_ONOFF_summary = pd.DataFrame.from_dict(dic_ONOFF_summary).T
    df_ONOFF_summary.columns = columns_name
    
    return df_ONOFF_summary

# 先独立运行完seperate session,再进行后面的分析
只需要参考一只小鼠的处理流程。不同小鼠除了文件保存路径不同外其他代码相同。

# Separate sessions & Save for later use

In [42]:
root_path = 'F:/AAA-RXC'
data_path = os.path.join(root_path)
# F:/AAA-Social analysis/Data-Neuron Social/AAA-PL/1-PL04-SOCIAL

IL1_list = ['PL1-4sessions',]
WT_mouse = ['WT']*len(IL1_list)

session_ids = [1,2,3,4]

In [43]:
#分离不同的session。根据TRACE文件中的'time'列中的值，将不同的session区分开。
def separate_session(df, threshold=3):
    df = pd.read_csv(df, index_col=0)
    
    df.columns = [c.strip() for c in df.columns]  #去除TRACE文件标题列的空格

    df['index_col'] = pd.to_numeric(df.index, errors='coerce')
    check_points = [0]  # 初始化第一个分割点为0（DataFrame的起始位置）
    frame_list = df['index_col'].values.tolist()

    # 判断两行之间的时间差，大于threshold则视为新的session
    for i in range(len(frame_list) - 1):
        if (frame_list[i+1] - frame_list[i]) > threshold:
            check_points.append(frame_list[i+1])
    
    check_points.append(frame_list[-1] + 1)  # 添加最后一个分割点

    session_dfs = []
    for i in range(len(check_points) - 1):
        session_df = df[(df['index_col'] >= check_points[i]) & (df['index_col'] < check_points[i+1])]
        session_df.drop(columns='index_col', inplace=True)
        session_dfs.append(session_df)

    return session_dfs

In [44]:
for i in range(len(IL1_list)):
    
    mouseSummary = IL1_list[i]
    mouseName = mouseSummary[-11:-7]   ### mouse name selected from file name
    mouseType = WT_mouse[i]
    print("Processing ", mouseName+'-'+mouseType)
    
    trace = os.path.join(data_path, mouseSummary, "TRACE_Con.csv")
    df_traces = separate_session(trace)

    for session_id in session_ids:
        event = os.path.join(data_path, mouseSummary, f"event{session_id}.xlsx")
        #event = os.path.join(data_path, mouseSummary, f"event.xlsx")
        #onoff = os.path.join(data_path, mouseSummary, project+'ONOFF.csv')
    
        session = df_traces[session_id-1]
        session.index.name="Frame"
        session.index = pd.to_numeric(session.index)
        session_frame_label = add_event_label(session, event)

        session_frame_label.to_csv(os.path.join(data_path, mouseSummary, f"session_{session_id}_trace.csv"))

        print(mouseName, f"session_{session_id}_trace has shape:", session_frame_label.shape)

    print('Done')
    session_frame_label.head(2)   
    

Processing  1-4s-WT


C:\Users\DELL\AppData\Local\Temp\ipykernel_21284\2922488257.py:3: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(df, index_col=0)
C:\Users\DELL\AppData\Local\Temp\ipykernel_21284\2922488257.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  session_df.drop(columns='index_col', inplace=True)
C:\Users\DELL\AppData\Local\Temp\ipykernel_21284\2922488257.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.p

1-4s session_1_trace has shape: (9004, 82)
1-4s session_2_trace has shape: (6303, 82)
1-4s session_3_trace has shape: (6303, 82)
1-4s session_4_trace has shape: (6302, 82)
Done


# ONOFF Neurons_Classification

In [10]:
root_path = 'F:/AAA-RXC'
data_path = os.path.join(root_path)
IL1_list = ['PL1-4sessions',]
session_ids = [1,2,]

In [11]:
print('start')

for i in range(len(IL1_list)):
    
    mouse_name = IL1_list[i]
    mouseSummary = IL1_list[i]
    
    print('Processing', mouse_name)
    
    ### 创建存储每只mouse结果的目录 F:/AAA-Social analysis/ONOFF
    mouse_dir = os.path.join(root_path, 'Results', mouse_name)
    if not os.path.exists(mouse_dir):
        os.makedirs(mouse_dir)


    # 选择FPS
    if mouse_name.split('_')[0] in ['PL1-4sessions',]:
        fps = 15
    else:
        fps = 15
        
    #dfLabeled, session_1, session_2, session_3 = add_event_label(trace, event1, event2, event3)
    #session_list = [session_1, session_2, session_3]
    sessionName_list = ['session_1', 'session_2', 'session_3', 'session_4']
    
    #for session_index in range(len(sessionName_list)):
    for sessionID in session_ids:
            
            
            event = os.path.join(data_path, mouseSummary, f"event{sessionID}.xlsx")
            session = pd.read_csv(os.path.join(data_path, mouseSummary, f"session_{sessionID}_trace.csv"))
            #session = session_list[session_index]
            sessionName = sessionName_list[sessionID-1]

            df_sniff_on, sniff_on_list = status_ONOFF(session, 'sniff', 'ON', fps)
            df_sniff_off, sniff_off_list = status_ONOFF(session, 'sniff', 'OFF', fps)

            df_sniffed_on, sniffed_on_list = status_ONOFF(session, 'sniffed', 'ON', fps)
            df_sniffed_off, sniffed_off_list = status_ONOFF(session, 'sniffed', 'OFF', fps)

            df_freezing_on, freezing_on_list = status_ONOFF(session, 'freezing', 'ON', fps)
            df_freezing_off, freezing_off_list = status_ONOFF(session, 'freezing', 'OFF', fps)

            df_cs_on, cs_on_list = status_ONOFF(session, 'cs', 'ON', fps)
            df_cs_off, cs_off_list = status_ONOFF(session, 'cs', 'OFF', fps)

            
            all_ONOFF_neuron_list = [sniff_on_list, sniff_off_list, sniffed_on_list, sniffed_off_list, freezing_on_list, freezing_off_list, cs_on_list, cs_off_list]
            columns_name = ['sniff_ON', 'sniff_OFF', 'sniffed_ON', 'sniffed_OFF', 'freezing_ON', 'freezing_OFF', 'cs_ON', 'cs_OFF']
            df_ONOFF_summary = generate_ONOFF_Summary(session, all_ONOFF_neuron_list, columns_name)
            
            csv_file_path = os.path.join(mouse_dir, f"{mouse_name}_{sessionName}_ONOFF.csv")
            df_ONOFF_summary.to_csv(csv_file_path)
            print(f"Saved: {csv_file_path}")
       


start
Processing PL1-4sessions


C:\Users\DELL\AppData\Local\Temp\ipykernel_38808\880704656.py:166: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['Frame_Label'] = frame_label_list
C:\Users\DELL\AppData\Local\Temp\ipykernel_38808\880704656.py:166: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['Frame_Label'] = frame_label_list
C:\Users\DELL\AppData\Local\Temp\ipykernel_38808\880704656.py:166: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = val

Saved: F:/AAA-RXC\Results\PL1-4sessions\PL1-4sessions_session_1_ONOFF.csv


C:\Users\DELL\AppData\Local\Temp\ipykernel_38808\880704656.py:166: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['Frame_Label'] = frame_label_list
C:\Users\DELL\AppData\Local\Temp\ipykernel_38808\880704656.py:166: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['Frame_Label'] = frame_label_list
C:\Users\DELL\AppData\Local\Temp\ipykernel_38808\880704656.py:166: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = val

Saved: F:/AAA-RXC\Results\PL1-4sessions\PL1-4sessions_session_2_ONOFF.csv
